# 🔬 RAG Assistant — Advanced Extensions
## Evaluation · Visualization · Stress Testing · Multi-Doc Comparison

> **Prerequisite:** Run `Smart_Document_QA_Assistant.ipynb` first in the same Colab session.
> All variables (`chroma_collection`, `db_conn`, `gemini_ef`, etc.) are expected to already be in memory.

---

| Extension | What it adds |
|-----------|-------------|
| **EXT-1** | Chunk Inspector — visualize chunk lengths and overlap |
| **EXT-2** | Embedding Space Visualizer — 2D UMAP/PCA plot of stored vectors |
| **EXT-3** | Retrieval Evaluator — precision/recall on labelled test pairs |
| **EXT-4** | Batch Q&A Stress Test — run N questions, log all latencies |
| **EXT-5** | Hallucination Detector — score answer groundedness |
| **EXT-6** | Multi-document Comparison — which doc answers better? |
| **EXT-7** | SQLite Analytics Dashboard — Gradio-powered log explorer |
| **EXT-8** | Export Q&A Report — save session as formatted TXT report |

---
## 📦 Install Additional Extension Dependencies

In [ ]:
# ============================================================
# EXT CELL 0 — Install Extra Packages for Extensions
# ============================================================

!pip install -q umap-learn          # UMAP dimensionality reduction
!pip install -q scikit-learn        # PCA + cosine similarity metrics
!pip install -q matplotlib          # Plotting
!pip install -q seaborn             # Statistical plots
!pip install -q plotly              # Interactive plots
!pip install -q rich                # Pretty terminal tables

print("✅ Extension dependencies installed.")

In [ ]:
# ============================================================
# EXT CELL 0b — Extension Imports
# ============================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from sklearn.decomposition import PCA
from sklearn.preprocessing import normalize
from sklearn.metrics.pairwise import cosine_similarity
import json, time, uuid, textwrap
from rich.console import Console
from rich.table import Table
from rich import box

console = Console()
sns.set_theme(style="darkgrid", palette="muted")
plt.rcParams.update({"figure.facecolor": "#1e293b", "axes.facecolor": "#0f172a",
                     "text.color": "#e2e8f0", "axes.labelcolor": "#94a3b8",
                     "xtick.color": "#94a3b8", "ytick.color": "#94a3b8",
                     "axes.edgecolor": "#334155", "grid.color": "#1e293b"})

print("✅ Extension imports ready.")

---
## 📊 EXT-1 — Chunk Inspector

Visualize the **distribution of chunk lengths** and **overlap coverage** for all documents in ChromaDB.
Helps verify chunking parameters are producing well-sized segments.

In [ ]:
# ============================================================
# EXT-1 — Chunk Length Distribution Inspector
# ============================================================

def inspect_chunks(collection) -> pd.DataFrame:
    """
    Pull all documents from ChromaDB and compute per-chunk statistics.
    Returns a DataFrame with: chunk_id, source, char_length, word_count.
    """
    result = collection.get(include=["documents", "metadatas"])

    records = []
    for doc_id, text, meta in zip(
        result["ids"], result["documents"], result["metadatas"]
    ):
        records.append({
            "chunk_id"   : doc_id,
            "source"     : meta.get("source", "unknown"),
            "chunk_idx"  : meta.get("chunk_idx", 0),
            "char_length": len(text),
            "word_count" : len(text.split()),
            "preview"    : text[:80].replace("\n", " ")
        })

    df = pd.DataFrame(records).sort_values(["source", "chunk_idx"])
    return df


# ── Load chunk data ───────────────────────────────────────
df_chunks = inspect_chunks(chroma_collection)

# ── Print summary ─────────────────────────────────────────
print(f"📊 Total chunks in store : {len(df_chunks)}")
print(f"   Unique sources        : {df_chunks['source'].nunique()}")
print(f"   Avg char length       : {df_chunks['char_length'].mean():.0f}")
print(f"   Min / Max char length : {df_chunks['char_length'].min()} / {df_chunks['char_length'].max()}")
print(f"   Avg word count        : {df_chunks['word_count'].mean():.0f}")

# ── Rich table: per-source breakdown ─────────────────────
tbl = Table(title="Per-Source Chunk Stats", box=box.ROUNDED, style="bold white")
tbl.add_column("Source",    style="cyan")
tbl.add_column("Chunks",   justify="right", style="yellow")
tbl.add_column("Avg Chars",justify="right", style="green")
tbl.add_column("Min",      justify="right")
tbl.add_column("Max",      justify="right")

for src, grp in df_chunks.groupby("source"):
    tbl.add_row(
        src,
        str(len(grp)),
        f"{grp['char_length'].mean():.0f}",
        str(grp['char_length'].min()),
        str(grp['char_length'].max())
    )
console.print(tbl)

# ── Plots ─────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle("Chunk Analysis", color="#e2e8f0", fontsize=14, fontweight="bold")

# Histogram: char lengths
axes[0].hist(df_chunks["char_length"], bins=30, color="#7c3aed", edgecolor="#1e293b", alpha=0.85)
axes[0].axvline(CHUNK_SIZE, color="#f59e0b", linewidth=2, linestyle="--", label=f"Target ({CHUNK_SIZE})")
axes[0].set_title("Character Length Distribution", color="#e2e8f0")
axes[0].set_xlabel("Characters")
axes[0].set_ylabel("Chunk Count")
axes[0].legend()

# Box plot: per source
sources = df_chunks["source"].unique()
data_by_src = [df_chunks[df_chunks["source"]==s]["char_length"].values for s in sources]
bp = axes[1].boxplot(data_by_src, patch_artist=True,
                     boxprops=dict(facecolor="#2563eb", alpha=0.7),
                     medianprops=dict(color="#f59e0b", linewidth=2))
axes[1].set_xticklabels([s[:20] for s in sources], rotation=15, ha="right")
axes[1].set_title("Char Length by Source", color="#e2e8f0")
axes[1].set_ylabel("Characters")

# Word count CDF
sorted_wc = np.sort(df_chunks["word_count"])
cdf = np.arange(1, len(sorted_wc)+1) / len(sorted_wc)
axes[2].plot(sorted_wc, cdf, color="#10b981", linewidth=2)
axes[2].fill_between(sorted_wc, cdf, alpha=0.15, color="#10b981")
axes[2].set_title("Word Count CDF", color="#e2e8f0")
axes[2].set_xlabel("Words per Chunk")
axes[2].set_ylabel("Cumulative %")

plt.tight_layout()
plt.savefig("/content/chunk_analysis.png", dpi=150, bbox_inches="tight")
plt.show()
print("\n📊 Plot saved to /content/chunk_analysis.png")

---
## 🌌 EXT-2 — Embedding Space Visualizer (PCA + Interactive Plotly)

Reduce 768-dimensional embedding vectors to **2D via PCA** and plot them interactively.
Chunks from the same document should cluster together — validating semantic coherence.

In [ ]:
# ============================================================
# EXT-2 — Embedding Space Visualizer (PCA)
# ============================================================

def visualize_embedding_space(collection, max_chunks: int = 300) -> None:
    """
    Fetch stored embeddings from ChromaDB, reduce to 2D with PCA,
    and render an interactive Plotly scatter plot.

    Parameters
    ----------
    collection : ChromaDB collection
    max_chunks : Cap at this many chunks to keep the plot readable.
    """
    print("⏳ Fetching embeddings from ChromaDB...")
    result = collection.get(
        include=["embeddings", "documents", "metadatas"],
        limit=max_chunks
    )

    embeddings = np.array(result["embeddings"])  # shape: (N, 768)
    documents  = result["documents"]
    metadatas  = result["metadatas"]
    sources    = [m.get("source", "unknown") for m in metadatas]
    previews   = [d[:100].replace("\n", " ") for d in documents]

    print(f"   {len(embeddings)} embedding vectors fetched (dim={embeddings.shape[1]})")

    # ── PCA: 768D → 2D ────────────────────────────────────
    print("🔄 Applying PCA (768D → 2D)...")
    pca = PCA(n_components=2, random_state=42)
    coords_2d = pca.fit_transform(normalize(embeddings))  # L2-normalize first

    explained = pca.explained_variance_ratio_
    print(f"   Variance explained: PC1={explained[0]:.1%}, PC2={explained[1]:.1%}")

    # ── Build DataFrame ────────────────────────────────────
    df_viz = pd.DataFrame({
        "x"      : coords_2d[:, 0],
        "y"      : coords_2d[:, 1],
        "source" : sources,
        "preview": previews
    })

    # ── Plotly Interactive Scatter ─────────────────────────
    fig = px.scatter(
        df_viz,
        x="x", y="y",
        color="source",
        hover_data={"preview": True, "x": False, "y": False},
        title=f"Embedding Space — PCA 2D Projection<br>"
              f"<sup>PC1={explained[0]:.1%} variance, PC2={explained[1]:.1%} variance</sup>",
        labels={"x": f"PC1 ({explained[0]:.1%})", "y": f"PC2 ({explained[1]:.1%})"},
        template="plotly_dark",
        color_discrete_sequence=px.colors.qualitative.Vivid,
        opacity=0.8
    )

    fig.update_traces(
        marker=dict(size=9, line=dict(width=0.5, color="white"))
    )
    fig.update_layout(
        width=900, height=600,
        legend_title_text="Document Source",
        plot_bgcolor="#0f172a",
        paper_bgcolor="#1e293b",
        font_color="#e2e8f0"
    )

    fig.show()

    # ── Variance explained bar chart ───────────────────────
    pca_full = PCA(n_components=min(20, embeddings.shape[0]), random_state=42)
    pca_full.fit(normalize(embeddings))
    cumvar = np.cumsum(pca_full.explained_variance_ratio_)

    fig2 = go.Figure()
    fig2.add_bar(
        x=list(range(1, len(pca_full.explained_variance_ratio_)+1)),
        y=pca_full.explained_variance_ratio_,
        name="Per-component",
        marker_color="#7c3aed"
    )
    fig2.add_scatter(
        x=list(range(1, len(cumvar)+1)),
        y=cumvar,
        name="Cumulative",
        line=dict(color="#f59e0b", width=2)
    )
    fig2.update_layout(
        title="PCA Explained Variance",
        xaxis_title="Principal Component",
        yaxis_title="Variance Explained",
        template="plotly_dark",
        width=800, height=350,
        paper_bgcolor="#1e293b",
        font_color="#e2e8f0"
    )
    fig2.show()


# ── Run visualizer ────────────────────────────────────────
if chroma_collection.count() > 0:
    visualize_embedding_space(chroma_collection)
else:
    print("⚠️  No chunks in ChromaDB. Run the main notebook ingestion first.")

---
## 🎯 EXT-3 — Retrieval Evaluator

Measure **retrieval quality** using labelled question-answer pairs.
Computes Hit Rate @ K and Mean Reciprocal Rank (MRR) — standard RAG evaluation metrics.

In [ ]:
# ============================================================
# EXT-3 — Retrieval Quality Evaluator (Hit Rate @K + MRR)
# ============================================================

# ── Ground-truth test set ─────────────────────────────────
# Each entry: (question, expected_keyword_in_retrieved_chunk)
# In production, replace with human-labelled (question, chunk_id) pairs.

EVAL_PAIRS = [
    ("What is Retrieval-Augmented Generation?",
     "Retrieval-Augmented Generation"),

    ("Who coined the term Artificial Intelligence?",
     "John McCarthy"),

    ("What is supervised learning?",
     "labelled data"),

    ("What are the layers in a neural network?",
     "Input Layer"),

    ("What is the difference between narrow AI and general AI?",
     "Narrow AI"),

    ("How does reinforcement learning work?",
     "cumulative reward"),

    ("What is a Convolutional Neural Network used for?",
     "image recognition"),

    ("What is sentiment analysis?",
     "Sentiment Analysis"),

    ("What ethical issues does AI raise?",
     "Bias and Fairness"),

    ("What economic impact will AI have by 2030?",
     "15.7 trillion"),
]


def evaluate_retrieval(
    eval_pairs: list,
    collection,
    top_k: int = TOP_K_RESULTS
) -> dict:
    """
    Evaluate retrieval quality on labelled question-keyword pairs.

    Metrics
    -------
    Hit Rate @K  : fraction of queries where the expected keyword
                   appears in at least one of the top-K retrieved chunks.
    MRR          : Mean Reciprocal Rank — how high the first correct
                   chunk ranks on average (1.0 = always top-1).
    Avg Latency  : Mean retrieval time in milliseconds.
    """
    results = []

    print(f"🎯 Evaluating {len(eval_pairs)} question-keyword pairs (top_k={top_k})...\n")

    for q, keyword in eval_pairs:
        t0 = time.time()
        try:
            chunks = retrieve_relevant_chunks(q, collection, top_k=top_k)
        except Exception as e:
            print(f"   ❌ Error for '{q[:50]}': {e}")
            continue

        latency = (time.time() - t0) * 1000

        # Find first rank where keyword appears (1-indexed)
        hit_rank = None
        for rank, chunk in enumerate(chunks, start=1):
            if keyword.lower() in chunk["text"].lower():
                hit_rank = rank
                break

        results.append({
            "question"   : q,
            "keyword"    : keyword,
            "hit"        : hit_rank is not None,
            "hit_rank"   : hit_rank,
            "rr"         : 1.0 / hit_rank if hit_rank else 0.0,
            "latency_ms" : latency,
            "top_sources": [c["source"] for c in chunks[:3]]
        })

        status = f"✅ Rank {hit_rank}" if hit_rank else "❌ Miss"
        print(f"   {status} | '{q[:55]}'")
        time.sleep(0.5)  # Rate limit

    # ── Compute aggregate metrics ──────────────────────────
    df_eval = pd.DataFrame(results)
    hit_rate = df_eval["hit"].mean()
    mrr      = df_eval["rr"].mean()
    avg_lat  = df_eval["latency_ms"].mean()

    print(f"\n{'='*55}")
    print(f"📊 RETRIEVAL EVALUATION RESULTS")
    print(f"{'='*55}")
    print(f"   Hit Rate @{top_k} : {hit_rate:.1%}  ({df_eval['hit'].sum()}/{len(df_eval)} hits)")
    print(f"   MRR          : {mrr:.3f}  (1.0 = perfect top-1)")
    print(f"   Avg Latency  : {avg_lat:.0f} ms")
    print(f"{'='*55}")

    # ── Bar chart ──────────────────────────────────────────
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    fig.suptitle("Retrieval Evaluation", color="#e2e8f0", fontsize=14, fontweight="bold")

    # Hit vs Miss
    counts = df_eval["hit"].value_counts()
    colors = ["#10b981" if k else "#ef4444" for k in counts.index]
    axes[0].bar(["Hit", "Miss"], [counts.get(True, 0), counts.get(False, 0)],
                color=["#10b981", "#ef4444"], edgecolor="#1e293b")
    axes[0].set_title(f"Hit Rate @{top_k}: {hit_rate:.1%}", color="#e2e8f0")
    axes[0].set_ylabel("Count")

    # Rank distribution (hits only)
    hit_ranks = df_eval[df_eval["hit"]]["hit_rank"]
    if not hit_ranks.empty:
        rank_counts = hit_ranks.value_counts().sort_index()
        axes[1].bar(rank_counts.index, rank_counts.values,
                    color="#7c3aed", edgecolor="#1e293b")
        axes[1].set_title("Hit Rank Distribution", color="#e2e8f0")
        axes[1].set_xlabel("Rank (1 = Top Result)")
        axes[1].set_ylabel("Count")
        axes[1].set_xticks(range(1, top_k+1))

    plt.tight_layout()
    plt.savefig("/content/retrieval_eval.png", dpi=150, bbox_inches="tight")
    plt.show()

    return {"hit_rate": hit_rate, "mrr": mrr, "avg_latency_ms": avg_lat, "df": df_eval}


# ── Run evaluation ────────────────────────────────────────
eval_results = evaluate_retrieval(EVAL_PAIRS, chroma_collection)

---
## ⚡ EXT-4 — Batch Q&A Stress Test

Run **N questions automatically**, collect all answers and latencies, then plot the results.
Useful for identifying slow queries or failures under load.

In [ ]:
# ============================================================
# EXT-4 — Batch Q&A Stress Test with Latency Analysis
# ============================================================

BATCH_QUESTIONS = [
    "What is Artificial Intelligence?",
    "Explain deep learning in simple terms.",
    "What are Large Language Models?",
    "How does RAG reduce hallucinations?",
    "What is the Transformer architecture?",
    "Describe the difference between ML and AI.",
    "What are the applications of NLP?",
    "What is Named Entity Recognition?",
    "How does unsupervised learning work?",
    "What is superintelligent AI?",
]


def run_batch_stress_test(
    questions: list,
    collection,
    db_conn,
    delay_between: float = 1.0
) -> pd.DataFrame:
    """
    Run all questions through the full RAG pipeline.
    Logs latency breakdown: embed_ms, retrieve_ms, llm_ms, total_ms.
    Returns a DataFrame of all results.
    """
    session = "stress_" + str(uuid.uuid4())[:6]
    records = []

    print(f"⚡ Running {len(questions)} questions (session: {session})...\n")

    for i, q in enumerate(questions, 1):
        print(f"   [{i:02d}/{len(questions)}] {q[:60]}")

        t_total = time.time()

        # Time: embed
        t0 = time.time()
        qvec = get_query_embedding(q)
        embed_ms = (time.time() - t0) * 1000

        # Time: retrieve
        t0 = time.time()
        chunks = retrieve_relevant_chunks(q, collection)
        retrieve_ms = (time.time() - t0) * 1000

        # Time: LLM
        t0 = time.time()
        try:
            answer, _ = generate_answer(q, collection, db_conn, session)
            status = "ok"
        except Exception as e:
            answer = str(e)
            status = "error"
        llm_ms = (time.time() - t0) * 1000
        total_ms = (time.time() - t_total) * 1000

        records.append({
            "question"   : q,
            "status"     : status,
            "embed_ms"   : round(embed_ms, 1),
            "retrieve_ms": round(retrieve_ms, 1),
            "llm_ms"     : round(llm_ms, 1),
            "total_ms"   : round(total_ms, 1),
            "answer_len" : len(answer),
        })

        time.sleep(delay_between)

    df = pd.DataFrame(records)

    # ── Summary ───────────────────────────────────────────
    print(f"\n{'='*55}")
    print("📊 STRESS TEST SUMMARY")
    print(f"{'='*55}")
    print(f"   Total questions  : {len(df)}")
    print(f"   Success rate     : {(df['status']=='ok').mean():.1%}")
    print(f"   Avg total latency: {df['total_ms'].mean():.0f} ms")
    print(f"   Avg embed time   : {df['embed_ms'].mean():.0f} ms")
    print(f"   Avg retrieve time: {df['retrieve_ms'].mean():.0f} ms")
    print(f"   Avg LLM time     : {df['llm_ms'].mean():.0f} ms")
    print(f"   Slowest query    : {df['total_ms'].max():.0f} ms")
    print(f"{'='*55}")

    # ── Latency breakdown stacked bar ─────────────────────
    fig, ax = plt.subplots(figsize=(14, 5))
    x = range(len(df))
    ax.bar(x, df["embed_ms"],    label="Embed",    color="#7c3aed", alpha=0.9)
    ax.bar(x, df["retrieve_ms"], label="Retrieve", color="#2563eb", alpha=0.9,
           bottom=df["embed_ms"])
    ax.bar(x, df["llm_ms"],     label="LLM",      color="#10b981", alpha=0.9,
           bottom=df["embed_ms"]+df["retrieve_ms"])

    ax.set_xticks(list(x))
    ax.set_xticklabels([f"Q{i+1}" for i in x])
    ax.set_title("Latency Breakdown per Query", color="#e2e8f0", fontsize=13)
    ax.set_ylabel("Milliseconds")
    ax.legend()
    ax.axhline(df["total_ms"].mean(), color="#f59e0b", linewidth=1.5,
               linestyle="--", label="Avg Total")

    plt.tight_layout()
    plt.savefig("/content/stress_test_latency.png", dpi=150, bbox_inches="tight")
    plt.show()

    return df


df_stress = run_batch_stress_test(BATCH_QUESTIONS, chroma_collection, db_conn)
display(df_stress)

---
## 🧪 EXT-5 — Groundedness / Hallucination Detector

Use **cosine similarity** between the LLM answer embedding and the retrieved context embeddings
to estimate how grounded the answer is in source material.

Score interpretation:
- **> 0.85** — Highly grounded ✅
- **0.65–0.85** — Partially grounded ⚠️
- **< 0.65** — Potentially hallucinated ❌

In [ ]:
# ============================================================
# EXT-5 — Groundedness Score (Hallucination Detector)
# ============================================================

def compute_groundedness(
    answer: str,
    retrieved_chunks: list[dict]
) -> dict:
    """
    Embed the answer and each retrieved chunk.
    Compute cosine similarity: answer vs each chunk.
    The max similarity is the 'groundedness score'.

    Returns dict with: max_score, avg_score, per_chunk_scores.
    """
    # Embed answer
    ans_vec = np.array(get_query_embedding(answer)).reshape(1, -1)

    # Embed all chunks
    chunk_texts = [c["text"] for c in retrieved_chunks]
    chunk_vecs  = np.array(gemini_ef(chunk_texts))  # shape: (K, 768)

    # Cosine similarity
    sims = cosine_similarity(ans_vec, chunk_vecs)[0]  # shape: (K,)

    per_chunk = [
        {"source": c["source"], "score": round(float(s), 4)}
        for c, s in zip(retrieved_chunks, sims)
    ]

    return {
        "max_score"        : round(float(sims.max()), 4),
        "avg_score"        : round(float(sims.mean()), 4),
        "per_chunk_scores" : per_chunk
    }


def interpret_score(score: float) -> str:
    if score > 0.85: return "✅ Highly grounded"
    if score > 0.65: return "⚠️  Partially grounded"
    return "❌ Potentially hallucinated"


# ── Test on 3 questions ───────────────────────────────────
TEST_GROUNDING_QS = [
    "What is the economic impact of AI by 2030?",
    "Who invented the lightbulb?",          # Not in the document — expect low score
    "What is a Transformer model in NLP?",
]

session_g = "ground_" + str(uuid.uuid4())[:6]
grounding_results = []

for q in TEST_GROUNDING_QS:
    print(f"\n❓ Query: {q}")
    try:
        answer, chunks = generate_answer(q, chroma_collection, db_conn, session_g)
        g = compute_groundedness(answer, chunks)

        grounding_results.append({"question": q, **g})

        print(f"   💬 Answer (preview): {answer[:120]}...")
        print(f"   📊 Groundedness Score: {g['max_score']} — {interpret_score(g['max_score'])}")
        for pc in g["per_chunk_scores"]:
            print(f"      └── {pc['source']}: {pc['score']}")

    except Exception as e:
        print(f"   ❌ Error: {e}")

    time.sleep(1.5)

# ── Groundedness bar chart ────────────────────────────────
if grounding_results:
    df_g = pd.DataFrame(grounding_results)
    colors = ["#10b981" if s > 0.85 else ("#f59e0b" if s > 0.65 else "#ef4444")
              for s in df_g["max_score"]]

    fig, ax = plt.subplots(figsize=(10, 4))
    bars = ax.barh(
        [f"Q{i+1}: {q[:40]}..." for i, q in enumerate(df_g["question"])],
        df_g["max_score"],
        color=colors, edgecolor="#1e293b"
    )
    ax.axvline(0.85, color="#10b981", linewidth=1.5, linestyle="--", label="Grounded threshold")
    ax.axvline(0.65, color="#f59e0b", linewidth=1.5, linestyle="--", label="Partial threshold")
    ax.set_xlim(0, 1)
    ax.set_title("Answer Groundedness Scores", color="#e2e8f0", fontsize=13)
    ax.set_xlabel("Max Cosine Similarity (Answer vs Retrieved Chunks)")
    ax.legend()
    plt.tight_layout()
    plt.savefig("/content/groundedness_scores.png", dpi=150, bbox_inches="tight")
    plt.show()

---
## 📋 EXT-6 — SQLite Analytics Dashboard (Gradio)

A **standalone Gradio dashboard** for exploring query logs stored in SQLite.
Run this after you've asked some questions to see full analytics.

In [ ]:
# ============================================================
# EXT-6 — SQLite Analytics Dashboard
# ============================================================

import gradio as gr
import pandas as pd

def get_summary_stats():
    df = pd.read_sql("SELECT * FROM query_logs", db_conn)
    if df.empty:
        return "No data yet.", "", ""

    stats = (
        f"**Total Queries:** {len(df)}  \n"
        f"**Unique Sessions:** {df['session_id'].nunique()}  \n"
        f"**Avg Latency:** {df['latency_ms'].mean():.0f} ms  \n"
        f"**Min / Max Latency:** {df['latency_ms'].min():.0f} / {df['latency_ms'].max():.0f} ms  \n"
        f"**Date Range:** {df['timestamp'].min()[:19]} → {df['timestamp'].max()[:19]}"
    )

    recent = df.tail(20)[["id","timestamp","user_query","source_docs","latency_ms"]]
    recent["timestamp"] = recent["timestamp"].str[:19]
    recent["user_query"] = recent["user_query"].str[:70]

    docs = pd.read_sql("SELECT * FROM document_registry", db_conn)
    docs_md = docs.to_markdown(index=False) if not docs.empty else "No documents registered."

    return stats, recent.to_markdown(index=False), docs_md


with gr.Blocks(
    title="RAG Analytics Dashboard",
    theme=gr.themes.Soft(primary_hue="violet", neutral_hue="slate")
) as dash:

    gr.HTML("<h2 style='text-align:center'>📊 RAG Analytics Dashboard</h2>")
    gr.HTML("<p style='text-align:center;color:#94a3b8'>SQLite Query Log Explorer</p>")

    refresh = gr.Button("🔄 Load / Refresh", variant="primary")

    with gr.Row():
        summary_box = gr.Markdown("_Click Refresh to load stats._")

    with gr.Accordion("📋 Recent Queries (last 20)", open=True):
        query_table = gr.Markdown("")

    with gr.Accordion("📄 Document Registry", open=True):
        doc_table = gr.Markdown("")

    refresh.click(
        fn=get_summary_stats,
        inputs=[],
        outputs=[summary_box, query_table, doc_table]
    )

dash.launch(share=True, quiet=True)

---
## 📄 EXT-7 — Export Full Q&A Session Report

Export the entire session — all queries, answers, sources and latencies — as a formatted **text report** you can download.

In [ ]:
# ============================================================
# EXT-7 — Export Q&A Session as Text Report
# ============================================================

from google.colab import files as colab_files
import json

def export_session_report(output_path: str = "/content/QA_Session_Report.txt") -> str:
    """
    Read all query_logs from SQLite and write a formatted report.
    Returns the path of the exported file.
    """
    df = pd.read_sql("SELECT * FROM query_logs ORDER BY id ASC", db_conn)

    if df.empty:
        print("⚠️  No queries found in the database.")
        return ""

    lines = [
        "=" * 70,
        "  SMART DOCUMENT Q&A ASSISTANT — SESSION REPORT",
        "=" * 70,
        f"  Generated   : {datetime.datetime.utcnow().isoformat()} UTC",
        f"  Total Queries: {len(df)}",
        f"  Avg Latency  : {df['latency_ms'].mean():.0f} ms",
        f"  LLM Model    : {LLM_MODEL}",
        f"  Embed Model  : {EMBEDDING_MODEL}",
        "=" * 70,
        ""
    ]

    for _, row in df.iterrows():
        lines += [
            f"{'─'*70}",
            f"Query #{row['id']:03d}  [{row['timestamp'][:19]}]  Session: {row['session_id']}",
            f"{'─'*70}",
            f"QUESTION:",
            f"  {row['user_query']}",
            "",
            f"ANSWER:",
        ]
        # Wrap answer at 70 chars
        for para in row['llm_answer'].split("\n"):
            lines += textwrap.wrap(para, width=68, initial_indent="  ", subsequent_indent="  ") or [""]

        lines += [
            "",
            f"SOURCES  : {row['source_docs']}",
            f"LATENCY  : {row['latency_ms']:.0f} ms",
            ""
        ]

        # Append retrieved chunks (summarised)
        try:
            chunks = json.loads(row['retrieved_chunks'])
            lines.append("RETRIEVED CONTEXT:")
            for i, chunk in enumerate(chunks[:3], 1):
                preview = chunk[:200].replace("\n", " ")
                lines.append(f"  [{i}] {preview}...")
        except Exception:
            pass

        lines.append("")

    lines += ["=" * 70, "  END OF REPORT", "=" * 70]

    report_text = "\n".join(lines)
    with open(output_path, "w", encoding="utf-8") as f:
        f.write(report_text)

    print(f"✅ Report written: {output_path}")
    print(f"   {len(df)} queries | {len(report_text):,} characters")
    return output_path


# ── Export and download ───────────────────────────────────
report_path = export_session_report()
if report_path:
    print("\n📥 Downloading report...")
    colab_files.download(report_path)

---
## ✅ All Extensions Complete!

| Extension | Output File |
|-----------|------------|
| EXT-1 Chunk Inspector | `/content/chunk_analysis.png` |
| EXT-2 Embedding Visualizer | Interactive Plotly in notebook |
| EXT-3 Retrieval Evaluator | `/content/retrieval_eval.png` |
| EXT-4 Stress Test | `/content/stress_test_latency.png` |
| EXT-5 Groundedness | `/content/groundedness_scores.png` |
| EXT-6 Analytics Dashboard | Live Gradio URL |
| EXT-7 Session Report | `/content/QA_Session_Report.txt` (downloaded) |

---
*Advanced Extensions for Smart Document Q&A Assistant · AI Assessment*